# CoRe-TFM — Simple JMLR Robustness Notebook

This is the clean version to use in Colab. It runs **one robustness variant at a time** and keeps the interface simple: setup → config → preflight → run/resume → inspect.

There is no experiment queue, no AMD-specific setup, and no raw `exec()` execution. The proven Q1 benchmark cells are executed by Jupyter itself. Completed folds are checkpointed to Google Drive, so rerunning the same configuration resumes the run.

Use a Colab GPU runtime and create a Colab Secret named `TABPFN_TOKEN`.


## Cell 1 — setup

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/core-tfm')
if not (ROOT/'.git').exists():
    subprocess.run(['git','clone','https://github.com/bnssaanirudh/core-tfm.git',str(ROOT)], check=True)
else:
    subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)
    subprocess.run(['git','checkout','main'], cwd=ROOT, check=True)
    subprocess.run(['git','reset','--hard','origin/main'], cwd=ROOT, check=True)

HEAD = subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip()
print('Repository HEAD:', HEAD)

subprocess.run([
    sys.executable,'-m','pip','install','-q','-e','.[test]',
    'pyyaml','nbformat','nbclient','ipykernel'
], cwd=ROOT, check=True)

SRC = str(ROOT/'src')
if SRC not in sys.path:
    sys.path.insert(0,SRC)
os.chdir(ROOT)

import nbformat
from nbclient import NotebookClient
from core_tfm.robustness_runner import (
    load_notebook, patch_q1_notebook, fold_result_status, write_complete_marker,
)
print('Setup complete.')


## Cell 2 — configuration

Usually change only `SEED` and `TRAIN_LIMIT`. Keep the same values when resuming an interrupted run.

Multi-seed study: use seeds `11, 23, 42, 71, 101` with `TRAIN_LIMIT=256`.

Context-size study: use train limits `64, 128, 256, 512, 1024` with seeds `23, 42, 71`.


In [ ]:
SEED = 11
TRAIN_LIMIT = 256
TEST_LIMIT = 128
SHARD_MINUTES = 25

RUN_ID = f'simple_runs/seed_{SEED}_train_{TRAIN_LIMIT}'
OUTPUT_ROOT = Path('/content/drive/MyDrive/CoRe_TFM_Q1/core_tfm_jmlr_simple_v2')
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('RUN_ID      :', RUN_ID)
print('SEED        :', SEED)
print('TRAIN_LIMIT :', TRAIN_LIMIT)
print('TEST_LIMIT  :', TEST_LIMIT)
print('RUN_DIR     :', RUN_DIR)


## Cell 3 — GPU, token and template preflight

In [ ]:
import torch
from google.colab import userdata

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is visible. Choose Runtime → Change runtime type → GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

token = (userdata.get('TABPFN_TOKEN') or '').strip()
if not token:
    raise RuntimeError('Create a Colab Secret named TABPFN_TOKEN and enable notebook access.')
os.environ['TABPFN_TOKEN'] = token
os.environ['TABPFN_NO_BROWSER'] = '1'
print('TABPFN_TOKEN: present')

TEMPLATE_PATH = ROOT/'notebooks'/'CoRe_TFM_Q1_FAST_COMPLETE_256_Colab.ipynb'
template = load_notebook(TEMPLATE_PATH)
patched = patch_q1_notebook(
    template,
    run_id=RUN_ID,
    seed=SEED,
    train_limit=TRAIN_LIMIT,
    test_limit=TEST_LIMIT,
    drive_base=str(OUTPUT_ROOT),
    session_minutes=SHARD_MINUTES,
    shard_minutes=SHARD_MINUTES,
    disable_controlled_replications=True,
    disable_selection_ablations=True,
    disable_validation_sensitivity=True,
)

# Keep the proven Q1 notebook only through Cell 12E.
kept = []
found_12e = False
for cell in patched['cells']:
    kept.append(cell)
    if cell.get('cell_type') == 'code':
        src = ''.join(cell.get('source', []))
        first = src.lstrip().splitlines()[0] if src.strip() else ''
        if 'Cell 12E' in first:
            found_12e = True
            break
assert found_12e, 'Could not locate Cell 12E in the Q1 template.'
patched['cells'] = kept

for cell in patched['cells']:
    cell['execution_count'] = None
    if cell.get('cell_type') == 'code':
        cell['outputs'] = []

generated_path = RUN_DIR/'generated_variant.ipynb'
executed_path = RUN_DIR/'executed_variant.ipynb'
nbformat.write(nbformat.from_dict(patched), generated_path)
print('Generated benchmark notebook:', generated_path)

before = fold_result_status(RUN_DIR)
print('Existing progress:')
print(json.dumps(before, indent=2))
print('PRECHECK PASS')


## Cell 4 — run / resume

This is the expensive cell. Jupyter executes the proven Q1 benchmark normally through Cell 12E. Each completed fold is checkpointed to Drive. If Colab disconnects, rerun this notebook with the same Cell 2 values.


In [ ]:
current = fold_result_status(RUN_DIR)
if current.get('complete'):
    print('This variant is already complete. Nothing to run.')
else:
    nb = nbformat.read(generated_path, as_version=4)
    client = NotebookClient(
        nb,
        timeout=None,
        kernel_name='python3',
        resources={'metadata': {'path': str(ROOT)}},
        allow_errors=False,
        record_timing=True,
    )
    try:
        client.execute()
    finally:
        nbformat.write(nb, executed_path)
        print('Executed notebook saved to:', executed_path)

after = fold_result_status(RUN_DIR)
print('CURRENT VARIANT STATUS:')
print(json.dumps(after, indent=2))
if after.get('last_failure'):
    print('LAST RECORDED FOLD FAILURE:')
    print(json.dumps(after['last_failure'], indent=2))
if after.get('complete') and not (RUN_DIR/'COMPLETE.json').exists():
    marker = write_complete_marker(RUN_DIR, {
        'execution_platform':'google_colab_cuda',
        'source_commit':HEAD,
        'seed':SEED,
        'requested_train_limit':TRAIN_LIMIT,
        'requested_test_limit':TEST_LIMIT,
        'executed_notebook':executed_path.name,
    })
    print('COMPLETE MARKER:', marker)


## Cell 5 — inspect results

In [ ]:
import pandas as pd

status = fold_result_status(RUN_DIR)
print(json.dumps(status, indent=2))

fold_file = RUN_DIR/'fold_results.csv'
if fold_file.exists() and fold_file.stat().st_size > 0:
    folds = pd.read_csv(fold_file)
    completed_cells = folds[['dataset','model','fold']].drop_duplicates().shape[0]
    print('Rows:', len(folds), '/ 1200')
    print('Completed fold cells:', completed_cells, '/ 150')
    display(
        folds.groupby(['model','method'],as_index=False)['joint_nll'].mean()
        .sort_values(['model','joint_nll'])
    )
else:
    print('No fold results have been written yet.')


## Cell 6 — next run

If the run is incomplete, keep the same Cell 2 settings and rerun. When it reaches **1200 rows / 150 fold cells**, change Cell 2 to the next seed or context size.

Do not change `SEED` or `TRAIN_LIMIT` inside a partially completed run directory.
